# S4b DRILL — frozen T3 rule: replication on 60 new ETFs (PRIMARY) + the stock universe (SECONDARY)

Pre-registration: `research/screens/S4B_DRILL_PREREG.md`. R1 is the rule, R2 the pass, R3 reporting, R4 validation
and lockbox, R5 the universes and the reading. Nothing here is tuned. Lockbox: everything is cut at load at
2024-12-31, Tiingo pulls use endDate 2024-12-31, and IS entries run 2011-01-03 .. 2023-12-29. The 2025-26 data on the
original 12 ETFs stays sealed (R4).

**Flow (run in this order; each step stops until the previous one is done):**
1. **Step 0** builds the PRIMARY universe by rule (R5a, rule v2) and prints it.
   - You review it and set `UNIVERSE_APPROVED = True`.
   - Paste the printed sha256 back; it's frozen in git (R5 freeze commit) and set as `UNIVERSE_SHA`.
   - Until then, nothing downstream runs.
2. **Part A** validates the null on the martingale world for BOTH universes (R5d): 16 seeds each, plus power. Each
   universe is gated on its own result.
3. **Part B** applies the frozen rule to real data, for the universes whose validation passed. It covers R2, the
   R3 tables, the R5c reading and the trial log.

**SECONDARY is SURVIVORSHIP-BIASED IN FAVOUR OF DIP-BUYING:** the corpus keeps survivors, whose dips recovered by
construction. A pass on stocks alone reads as "likely survivorship".

In [ ]:
# ===== Cell 0 — config (pack-2 + S4b + drill) =====
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass                                            # local smoke test: BUKOWSKI_DIR points at fixtures
import pandas as pd, numpy as np, glob, json, os, math, urllib.request
from pathlib import Path
from scipy import stats
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path(os.environ.get('BUKOWSKI_DIR', '/content/drive/MyDrive/Bukowski'))
REF_DIR  = DATA_DIR/'reference'
OUT_DIR  = DATA_DIR/'results'/'screens'; OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- LOCKBOX ----
CUT_DATE = pd.Timestamp('2024-12-31')     # every series cut at load; nothing after this is read, pulled or printed
import re, io, zipfile, hashlib, time
from numba import njit, prange

IS_START, PURGE = pd.Timestamp('2011-01-01'), pd.Timestamp('2023-12-29')   # entries IS_START..PURGE; holds resolve <= cut
Y0, NY = 2011, 13                                         # years 2011..2023
ERAS = {'2011-2014': (2011, 2014), '2015-2018': (2015, 2018), '2019-2023': (2019, 2023)}; RECENT = '2019-2023'
COST = 0.0015                                             # round trip
TOP_U2 = 300                                              # U2 = top 300 stocks by trailing 60-bar $vol (t-60..t-1)
S4_HOLD, S4_STOP = 10, 0.10                               # 10-bar time stop; 10% disaster stop
S5_HOLDS = [20, 60]
S6_MIN_NAMES = 5                                          # top decile, at least 5 names
N_DRAWS, TRIES, DRAW_BATCH = int(os.environ.get('PACK2_DRAWS', 200)), 8, 50
PCT_PASS, MIN_YEARS = 90.0, 9                             # verdict: null pct >= 90; >= 9/13 years positive
FH_CANDIDATES = {'S4': [60], 'S5': [20, 60]}              # fixed-horizon excess (Stage 2b v2 method)
CI_LEVEL, BOOT_REPS = 0.90, 2000
P2_DIR = OUT_DIR/'pack2'; CACHE = P2_DIR/'cache'; CACHE.mkdir(parents=True, exist_ok=True)
ASSET_FILE, U1_FILE, PROTO_FILE = P2_DIR/'asset_class.csv', P2_DIR/'u1_etfs.csv', P2_DIR/'pack2_protocol.json'
UNIVS = ['U1', 'U2']
# Reviewed 2026-10-03 (Step 0 output): Tiingo rows misclassify these; shared by both notebooks.
ASSET_OVERRIDES = {'PAM': 'Stock',     # Pampa Energia SA (ADR), listed as ETF
                   'BNY': 'Stock',     # BNY Mellon (ticker from BK, 2024); old BNY row was a fund
                   'ERO': 'Stock'}     # Ero Copper; old ERO row was an exchange-traded note
CONFIGS = ([dict(name=f'S4_{tg}_{ex}_{st}', screen='S4', family=f'S4_{tg}', trig=tg, mode=(0 if ex == 'X1' else 1),
                 hold=S4_HOLD, stop=(st == 'stop10'))
            for tg in ['T1', 'T2', 'T3'] for ex in ['X1', 'X2'] for st in ['nostop', 'stop10']]
           + [dict(name=f'S5_h{h}', screen='S5', family='S5', trig=None, mode=2, hold=h, stop=False) for h in S5_HOLDS])


# ---- S4b (decisions 2026-10-03) ----
S4B_DIR = OUT_DIR/'s4b'; S4B_DIR.mkdir(parents=True, exist_ok=True); S4B_PROTO = S4B_DIR/'s4b_protocol.json'
ETFS = ['SPY', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']
TRIGS = ['T1', 'T2', 'T3']
OUTCOMES = [('X1', 0, 10), ('X2', 1, 10), ('H5', 2, 5), ('H10', 2, 10), ('H20', 2, 20)]   # (name, sim mode, hold bars)
MIN_EVENTS_YEAR, MIN_QUAL_YEARS, Q_PASS = 10, 9, 90.0
S4B_BINDING = 64                                          # 2026-10-03: ONE binding 64-seed run (replaces the 16-seed FAIL)
S4B_SEEDS = int(os.environ.get('S4B_SEEDS', S4B_BINDING)); S4B_POWER_SEEDS = int(os.environ.get('S4B_POWER_SEEDS', 4))
M_STEPS, ON_FRAC, MKT_SD, MKT_MU = 12, 0.2, 0.01, 0.0003
PLANT_REV = 0.010                                         # power: +1.0%/bar x 3 bars after a > 2 sigma down close (~40 bp/event)
TRIAL_LOG, TRIAL_RET = OUT_DIR/'screens_trial_log.csv', OUT_DIR/'screens_trial_returns.csv'


# ---- S4b DRILL (S4B_DRILL_PREREG.md R1-R5; frozen 2026-10-03) ----
DRILL_DIR = OUT_DIR/'s4b_drill'; DCACHE = DRILL_DIR/'cache'; DCACHE.mkdir(parents=True, exist_ok=True)
REP_DIR = REF_DIR/'replication'; REP_DIR.mkdir(parents=True, exist_ok=True)
UNIV_FILE, DRILL_PROTO = DRILL_DIR/'replication_universe.csv', DRILL_DIR/'drill_protocol.json'
UNIV_RULE_FILE, UNIV_RULE_VERSION = DRILL_DIR/'replication_universe_rule.json', 'v2'   # v2: review fixes 2026-10-03
U1_ETFS = ['SPY', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 'XLP', 'XLRE', 'XLU', 'XLV', 'XLY']
N_ETF, MAX_COUNTRY, CORR_MAX, RANK_YEAR, MIN_BARS_2013 = 60, 20, 0.95, 2013, 200
US_EXCH = {'NYSE', 'NYSE ARCA', 'NYSE MKT', 'AMEX', 'NASDAQ', 'BATS'}
DRILL_EXITS = [('X1', 0, 10), ('X2', 1, 10)]               # frozen T3 rule, R1
Q_DRILL, FALSE_PASS_MAX = 95.0, 0.10
DRILL_SEEDS = int(os.environ.get('DRILL_SEEDS', 16)); DRILL_POWER_SEEDS = int(os.environ.get('DRILL_POWER_SEEDS', 4))
DRILL_PLANT = 0.010
UNIVERSE_APPROVED = False or os.environ.get('DRILL_APPROVED') == '1'   # set True after reviewing the printed list
UNIVERSE_SHA = os.environ.get('DRILL_UNIVERSE_SHA', '')    # FROZEN sha256 of replication_universe.csv (R5 freeze commit)
LEV_RE = re.compile(r'(\b-?[1-4](\.\d+)?x\b|ultra|\bbear\b|inverse|leveraged|\bvix\b|(?<!low )(?<!min )volatility'
                    r'|\bshort\b(?![- ](term|duration|maturity|treasury|bond)))', re.I)
NONEQ_RE = re.compile(r'(\bbond|treasury|\btreas\b|\bmuni|municipal|\btips\b|inflation[- ]protected|floating rate|'
                      r'senior loan|\bloan\b|preferred|mortgage|\bmbs\b|aggregate|high yield|corporate|\bcredit\b|'
                      r'fixed income|money market|t-bill|gold trust|gold shares|silver trust|physical|bullion|'
                      r'commodit|oil fund|gas fund|invesco db|powershares db|currencyshares|dollar index|futures|'
                      r'covered call|buy-?write|allocation|target date|managed futures|convertible|maturity|'
                      r'duration|ultra[- ]?short|short[- ]term)', re.I)
ETN_RE = re.compile(r'(\betns?\b|exchange[- ]traded notes?|\bnotes? due\b|ipath|etracs|\belements\b|velocityshares)', re.I)
FUND_RE = re.compile(r'(\betf\b|\bfund\b|\btrust\b|\bindex\b|portfolio|\bshares\b|ishares|spdr|vanguard|powershares|'
                     r'invesco|vaneck|market vectors|wisdomtree|first trust|global x|schwab|guggenheim|\balps\b|'
                     r'xtrackers|x-trackers|proshares|direxion|pimco|select sector)', re.I)
COUNTRY_RE = re.compile(r'(msci (?!usa)|emerging|europe|euro\b|eurozone|asia|pacific|eafe|international|\bintl\b|'
                        r'world ex|ex-us|ex us|ex-u\.s|global|china|japan|india|brazil|mexico|canada|germany|'
                        r'united kingdom|\bu\.k\.|australia|korea|taiwan|hong kong|singapore|south africa|russia|'
                        r'latin america|\bbric|frontier|developed markets|foreign|acwi|chile|turkey|indonesia|'
                        r'thailand|malaysia|philippines|vietnam|israel|italy|spain|france|switzerland|sweden|'
                        r'netherlands|belgium|austria|nordic|middle east|africa|peru|colombia|poland|greece|'
                        r'ireland|argentina|norway|qatar|saudi)', re.I)

In [ ]:
# ===== Cell 1 — core (pack-2 + S4b + drill) =====
def load_px(path):
    """Read OHLC(+V) and CUT at CUT_DATE before anything else touches it."""
    cols = pd.read_csv(path, nrows=0).columns
    use = [c for c in ['Date', 'Open', 'High', 'Low', 'Close', 'Volume'] if c in cols]
    d = pd.read_csv(path, usecols=use); d['Date'] = pd.to_datetime(d['Date'])
    d = d.sort_values('Date').drop_duplicates('Date')
    d = d[d['Date'] <= CUT_DATE].reset_index(drop=True)                  # LOCKBOX
    if 'Volume' not in d.columns: d['Volume'] = np.nan
    return d


def tiingo_token():
    token = os.environ.get('TIINGO_API_KEY')
    if not token:
        try:
            from google.colab import userdata; token = userdata.get('TIINGO_API_KEY')
        except Exception:
            token = None
    return token


def ref_path(tk):
    """RSP historically lives at DATA_DIR/RSP.csv or REF_DIR/RSP.csv; everything new goes to REF_DIR."""
    return DATA_DIR/f'{tk}.csv' if tk == 'RSP' and (DATA_DIR/'RSP.csv').exists() else REF_DIR/f'{tk}.csv'


def ensure_ref(tk):
    """One-off ADJUSTED Tiingo pull, only if missing. endDate = CUT_DATE so sealed rows never reach Drive."""
    path = ref_path(tk)
    if path.exists():
        print(f'{tk}: present at {path} — no pull'); return path
    token = tiingo_token()
    assert token, f'Set TIINGO_API_KEY (Colab secret or env) to pull {tk}.'
    url = (f'https://api.tiingo.com/tiingo/daily/{tk}/prices?startDate=2003-01-01'
           f'&endDate={CUT_DATE:%Y-%m-%d}&token={token}')
    js = pd.DataFrame(json.load(urllib.request.urlopen(url, timeout=60)))
    df = pd.DataFrame({'Date': pd.to_datetime(js['date']).dt.strftime('%Y-%m-%d'),
                       'Open': js['adjOpen'], 'High': js['adjHigh'], 'Low': js['adjLow'],
                       'Close': js['adjClose'], 'Volume': js['adjVolume']})
    df = df[pd.to_datetime(df['Date']) <= CUT_DATE]                     # belt and braces
    REF_DIR.mkdir(parents=True, exist_ok=True); df.to_csv(path, index=False)
    print(f'{tk}: pulled {len(df)} rows (adjusted) -> {path}')
    return path


def close_ret(d, cal=None):
    """Close-to-close simple return on the series' own rows (optionally restricted to a calendar first)."""
    s = d.set_index('Date')['Close'].astype(float)
    if cal is not None: s = s[s.index.isin(cal)]
    s = s.where(s > 0)
    return (s / s.shift(1) - 1).dropna()


# ---------------------------------------------------------------- panels (T x N), cut at load, cached
def load_panels():
    f_cache = CACHE/'panels_cut_2024-12-31.npz'
    if f_cache.exists():
        z = np.load(f_cache)
        cal = pd.DatetimeIndex(z['dates']); assert cal.max() <= CUT_DATE
        return [str(t) for t in z['tickers']], cal, {c: z[c].astype(np.float64) for c in 'OHLCV'}
    spy = load_px(DATA_DIR/'SPY.csv'); cal = pd.DatetimeIndex(spy['Date']); T = len(cal)
    arrs, tickers = {c: [] for c in 'OHLCV'}, []
    for f in sorted(glob.glob(f'{DATA_DIR}/*.csv')):
        try:
            d = load_px(f)
        except Exception:
            continue
        pos = cal.get_indexer(d['Date']); ok = pos >= 0
        if not ok.any(): continue
        tickers.append(Path(f).stem.upper())
        for c, col in zip('OHLCV', ['Open', 'High', 'Low', 'Close', 'Volume']):
            a = np.full(T, np.nan, np.float32); a[pos[ok]] = d[col].to_numpy(float)[ok]; arrs[c].append(a)
    P = {c: np.stack(arrs[c], axis=1) for c in 'OHLCV'}
    bad = ~(P['C'] > 0)
    for c in 'OHLC': P[c][bad] = np.nan
    np.savez(f_cache, tickers=np.array(tickers), dates=cal.values, **P)
    return tickers, cal, {c: P[c].astype(np.float64) for c in P}


def lag(X, n):
    Y = np.full_like(X, np.nan); Y[n:] = X[:-n]; return Y


def roll(X, w, how='mean', minp=None):
    r = pd.DataFrame(X).rolling(w, min_periods=minp)
    return getattr(r, how)().to_numpy()


def indicators(O, H, L, C, V):
    """All causal: values at row t use bars <= t; the *p ones use bars <= t-1."""
    I = {'S5': roll(C, 5), 'S50': roll(C, 50), 'S200': roll(C, 200)}
    C1 = lag(C, 1); d = C - C1
    au = pd.DataFrame(np.where(d > 0, d, 0.0)).where(np.isfinite(d)).ewm(alpha=0.5, adjust=False).mean().to_numpy()
    ad = pd.DataFrame(np.where(d < 0, -d, 0.0)).where(np.isfinite(d)).ewm(alpha=0.5, adjust=False).mean().to_numpy()
    tot = au + ad
    I['RSI'] = np.where(np.isfinite(d), np.where(tot > 0, 100 * au / np.where(tot > 0, tot, 1), 50.0), np.nan)
    TR = np.fmax(H - L, np.fmax(np.abs(H - C1), np.abs(L - C1)))
    I['ATRp'] = lag(roll(TR, 14), 1)
    I['VA50p'] = lag(roll(V, 50), 1)
    I['DV60p'] = lag(roll(C * V, 60), 1)
    I['LOW20p'] = roll(C1, 20, 'min')
    ON = np.log(O / C1); ON[~(np.abs(ON) < np.log(1.5))] = np.nan          # overnight bad prints dropped
    I['ON21'] = roll(ON, 21, 'sum', minp=18)
    return I


def universes(DV60p, is_stock, is_u1):
    T, N = DV60p.shape
    X = np.where(is_stock[None, :], DV60p, np.nan)
    U2 = np.zeros((T, N), bool)
    for t in range(T):
        idx = np.nonzero(np.isfinite(X[t]) & (X[t] > 0))[0]
        if len(idx) > TOP_U2: idx = idx[np.argpartition(-X[t, idx], TOP_U2 - 1)[:TOP_U2]]
        U2[t, idx] = True
    U1 = is_u1[None, :] & np.isfinite(DV60p) & (DV60p > 0)
    return {'U1': U1, 'U2': U2}


def fwd(C, h):
    F = np.full_like(C, np.nan); F[:-h] = C[h:] / C[:-h] - 1; return F


def build_ctx(O, H, L, C, V, is_stock, is_u1, i_spy):
    I = indicators(O, H, L, C, V)
    UN = universes(I['DV60p'], is_stock, is_u1)
    C1, C2, C3 = lag(C, 1), lag(C, 2), lag(C, 3)
    trend = (C > I['S200']) & (I['S50'] > I['S200'])
    TRIG = {'T1': I['RSI'] < 10, 'T2': (C < C1) & (C1 < C2) & (C2 < C3), 'T3': C < I['LOW20p']}
    quiet = np.abs(C - C1) < I['ATRp']; spike = V >= 3 * I['VA50p']; va = I['VA50p'] > 0
    NT = {k: np.ascontiguousarray(x.T) for k, x in dict(O=O, H=H, L=L, C=C, S5=I['S5'], RSI=I['RSI']).items()}
    hs = sorted({h for v in FH_CANDIDATES.values() for h in v})
    FW = {h: fwd(C, h) for h in hs}
    EWF = {(u, h): np.nanmean(np.where(UN[u], FW[h], np.nan), axis=1) for u in UNIVS for h in hs}
    return dict(I=I, UN=UN, trend=trend, TRIG=TRIG, s5=spike & quiet & va, s5pool=quiet & ~spike & va,
                NT=NT, FW=FW, EWF=EWF, C=C, i_spy=i_spy)


# ---------------------------------------------------------------- simulator (Stage 2 semantics)
@njit(cache=True)
def sim_one(O, H, L, C, S5, RSI, k, e, mode, hold, use_stop, stop_pct):
    """Enter at O[k,e]. Bar order: disaster stop (low) -> time stop (close of bar e+hold-1) -> exit signal at the
    close (mode 0: close > SMA5; 1: RSI2 > 70; 2: none), filled at the NEXT open. Data gap/end -> last close.
    Returns (exit bar, gross return, exit type: 0 = filled at that bar's open, 1 = at/near that bar's close)."""
    T = C.shape[1]
    if e >= T: return -1, np.nan, 0
    fill = O[k, e]
    if not (fill > 0.0): return -1, np.nan, 0
    stop = fill * (1.0 - stop_pct)
    last = e + hold - 1
    if last > T - 1: last = T - 1
    for i in range(e, last + 1):
        c = C[k, i]
        if not (c > 0.0):
            if i == e: return -1, np.nan, 0
            return i - 1, C[k, i - 1] / fill - 1.0, 1
        if use_stop and L[k, i] <= stop:
            o = O[k, i]
            px = stop if (i == e or not (o > 0.0)) else (o if o < stop else stop)
            return i, px / fill - 1.0, 1
        if i == last:
            return i, c / fill - 1.0, 1
        hit = False
        if mode == 0:
            hit = c > S5[k, i]
        elif mode == 1:
            hit = RSI[k, i] > 70.0
        if hit:
            o = O[k, i + 1]
            if o > 0.0: return i + 1, o / fill - 1.0, 0
            return i, c / fill - 1.0, 1
    return last, C[k, last] / fill - 1.0, 1


@njit(cache=True)
def run_rule(sig, O, H, L, C, S5, RSI, mode, hold, use_stop, stop_pct, t_lo, t_hi):
    """Signals at close t in [t_lo, t_hi] -> entry t+1. One position per ticker (next signal allowed at t >= exit bar)."""
    N = C.shape[0]; cap = 0
    for k in range(N):
        for t in range(t_lo, t_hi + 1):
            if sig[k, t]: cap += 1
    ok = np.empty(cap, np.int64); ot = np.empty(cap, np.int64); ox = np.empty(cap, np.int64); orr = np.empty(cap)
    oxt = np.empty(cap, np.int64)
    m = 0
    for k in range(N):
        free = 0
        for t in range(t_lo, t_hi + 1):
            if sig[k, t] and t >= free:
                x, r, xt = sim_one(O, H, L, C, S5, RSI, k, t + 1, mode, hold, use_stop, stop_pct)
                if x >= 0:
                    ok[m] = k; ot[m] = t; ox[m] = x; orr[m] = r; oxt[m] = xt; m += 1; free = x
    return ok[:m], ot[:m], ox[:m], orr[:m], oxt[:m]


@njit(cache=True)
def sim_null(O, C, kk, e, x, xt):
    """Duration-matched null trade on ticker kk: enter at O[kk,e], exit at the matched real trade's exit bar x with
    the same price type (0: open, 1: close). Missing exit price -> last close in [e, x]; none -> rejected (-1)."""
    fill = O[kk, e]
    if not (fill > 0.0): return -1.0
    px = O[kk, x] if xt == 0 else C[kk, x]
    j = x
    while not (px > 0.0):
        if j < e: return -1.0
        px = C[kk, j]; j -= 1
    return px / fill - 1.0


@njit(parallel=True, cache=True)
def null_draws(tr_t, tr_k, tr_x, tr_xt, ptr, pidx, U, O, C, cost, yidx, ny):
    """SELECTION null (DURATION-MATCHED): for each real trade (signal bar t, ticker k, exit bar x, exit type) draw a
    random OTHER ticker from the pool at t (same universe, same non-trigger condition), enter at its next open and exit
    at the SAME bar with the same price type. Only the ticker differs, so holding-time x market-drift cannot bias it
    (the own-exit version failed the martingale protocol: S4 families at pct 64-66, t ~3, with drift on).
    Non-overlap per draw. Returns per-draw, per-entry-year sums and counts of net returns."""
    D = U.shape[0]; n = tr_t.shape[0]; tries = U.shape[2]; N = C.shape[0]
    S = np.zeros((D, ny)); Cn = np.zeros((D, ny)); miss = np.zeros(D, np.int64)
    for d in prange(D):
        busy = np.zeros(N, np.int64)
        for i in range(n):
            t = tr_t[i]; a = ptr[t]; m = ptr[t + 1] - a; done = False
            if m > 0:
                for j in range(tries):
                    q = int(U[d, i, j] * m)
                    if q >= m: q = m - 1
                    kk = pidx[a + q]
                    if kk == tr_k[i] or busy[kk] > t: continue
                    r = sim_null(O, C, kk, t + 1, tr_x[i], tr_xt[i])
                    if r == -1.0: continue
                    busy[kk] = tr_x[i]; y = yidx[t + 1]
                    if y >= 0 and y < ny:
                        S[d, y] += r - cost; Cn[d, y] += 1.0
                    done = True; break
            if not done: miss[d] += 1
    return S, Cn, miss


def year_ci(x, blocks, reps, seed, level):
    """Year-block bootstrap of the mean: (CI lo, CI hi, q = 100 x share of bootstrap means > 0). Stage 2b v2."""
    x = np.asarray(x, float); ub, inv = np.unique(blocks, return_inverse=True)
    Sm = np.bincount(inv, weights=x, minlength=len(ub)); Cn = np.bincount(inv, minlength=len(ub)).astype(float)
    idx = np.random.default_rng(seed).integers(0, len(ub), (reps, len(ub))); mb = Sm[idx].sum(1) / Cn[idx].sum(1)
    a = (1 - level) / 2
    return float(np.quantile(mb, a)), float(np.quantile(mb, 1 - a)), 100.0 * float((mb > 0).mean())


def run_cfg(ctx, cfg, u, n_draws, seed, fh_list):
    """Real trades of one rule in one universe + selection null + fixed-horizon excess."""
    U = ctx['UN'][u]
    if cfg['screen'] == 'S4':
        sig, pool = ctx['trend'] & ctx['TRIG'][cfg['trig']] & U, ctx['trend'] & U
    else:
        sig, pool = ctx['s5'] & U, ctx['s5pool'] & U
    nt = ctx['NT']; T = U.shape[0]
    k, t, x, r, xt = run_rule(np.ascontiguousarray(sig.T), nt['O'], nt['H'], nt['L'], nt['C'], nt['S5'], nt['RSI'],
                              cfg['mode'], cfg['hold'], cfg['stop'], S4_STOP, T_LO, T_HI)
    o = np.argsort(t, kind='stable'); k, t, x, r, xt = k[o], t[o], x[o], r[o], xt[o]
    net = r - COST; yr = YEAR_T[t + 1] if len(t) else np.array([], int)
    out = dict(cfg=cfg['name'], screen=cfg['screen'], family=cfg['family'], universe=u, n=len(t), k=k, t=t, x=x,
               gross=r, net=net, year=yr)
    # selection null
    tt, kk = np.nonzero(pool)
    ptr = np.searchsorted(tt, np.arange(T + 1)).astype(np.int64); pidx = kk.astype(np.int64)
    Sd, Cd = np.zeros((n_draws, NY)), np.zeros((n_draws, NY)); miss = 0; rng = np.random.default_rng(seed)
    for b0 in range(0, n_draws, DRAW_BATCH):
        nb_ = min(DRAW_BATCH, n_draws - b0)
        Ub = rng.random((nb_, len(t), TRIES), dtype=np.float32)
        s_, c_, m_ = null_draws(t, k, x, xt, ptr, pidx, Ub, nt['O'], nt['C'], COST, YIDX, NY)
        Sd[b0:b0 + nb_], Cd[b0:b0 + nb_] = s_, c_; miss += int(m_.sum())
    with np.errstate(invalid='ignore', divide='ignore'):
        dm = Sd.sum(1) / Cd.sum(1)
        out['null_year_mean'] = Sd.sum(0) / Cd.sum(0)
    out['null_mean'] = float(np.nanmean(dm)) if len(t) else np.nan
    out['pct'] = float(100 * np.mean(dm < net.mean())) if len(t) and np.isfinite(dm).all() else np.nan
    out['null_miss_rate'] = miss / max(1, n_draws * len(t))
    lo_, hi_ = ERAS[RECENT]; era = (yr >= lo_) & (yr <= hi_)
    ey = np.arange(lo_ - Y0, hi_ - Y0 + 1)
    with np.errstate(invalid='ignore', divide='ignore'):
        out['era_exc'] = (float(net[era].mean() - Sd[:, ey].sum() / Cd[:, ey].sum()) if era.any() else np.nan)
    # fixed-horizon excess: close of the fill bar -> close h bars later, vs universe EW and SPY, same window
    out['fh'] = {}
    for h in fh_list:
        if not len(t): out['fh'][h] = None; continue
        e = t + 1; F = ctx['FW'][h]
        s_ = F[e, k]; ew = ctx['EWF'][(u, h)][e]; sp = F[e, ctx['i_spy']]
        res = {}
        for bm, b in (('ew', ew), ('spy', sp)):
            d_ = s_ - b; ok_ = np.isfinite(d_)
            ci = year_ci(d_[ok_], yr[ok_], BOOT_REPS, seed + h, CI_LEVEL) if ok_.sum() >= 20 else (np.nan,) * 3
            res[bm] = dict(n=int(ok_.sum()), mean=float(d_[ok_].mean()) if ok_.any() else np.nan,
                           lo=ci[0], hi=ci[1], q=ci[2])
        out['fh'][h] = res
    return out


def s6_monthly(ctx, u, cal):
    """Month-end formation: top decile (>= 5 names) by trailing 21-bar overnight-return sum; hold close->next
    month-end close, EW; minus all universe members over the same window. Turnover cost = replaced share x COST."""
    C, ON21, Um = ctx['C'], ctx['I']['ON21'], ctx['UN'][u]
    per = cal.to_period('M'); me = list(np.nonzero(per[:-1] != per[1:])[0]) + [len(cal) - 1]
    Cff = pd.DataFrame(C).ffill().to_numpy(); rows, prev = [], None
    for a, b in zip(me[:-1], me[1:]):
        if cal[b] < IS_START or cal[b] > PURGE: continue
        idx = np.nonzero(Um[a] & np.isfinite(ON21[a]) & (C[a] > 0))[0]
        if len(idx) < 2 * S6_MIN_NAMES: continue
        kk = max(int(math.ceil(len(idx) / 10)), S6_MIN_NAMES)
        top = idx[np.argsort(-ON21[a, idx], kind='stable')[:kk]]
        R_all = Cff[b, idx] / C[a, idx] - 1; R_top = Cff[b, top] / C[a, top] - 1
        gross = float(np.nanmean(R_top) - np.nanmean(R_all))
        turn = 1.0 if prev is None else 1 - len(set(top.tolist()) & prev) / kk; prev = set(top.tolist())
        rows.append(dict(month=str(cal[b].to_period('M')), year=cal[b].year, n_univ=len(idx), k=kk, gross=gross,
                         turnover=turn, net=gross - turn * COST))
    return pd.DataFrame(rows)


def s6_summary(M, seed=0):
    if M.empty: return dict(n=0)
    lo_, hi_ = ERAS[RECENT]; era = M[(M.year >= lo_) & (M.year <= hi_)]
    ci = year_ci(M['net'].to_numpy(), M['year'].to_numpy(), BOOT_REPS, seed, CI_LEVEL) if len(M) >= 20 else (np.nan,) * 3
    cig = year_ci(M['gross'].to_numpy(), M['year'].to_numpy(), BOOT_REPS, seed, CI_LEVEL) if len(M) >= 20 else (np.nan,) * 3
    yr = M.groupby('year')['net'].sum().reindex(range(Y0, Y0 + NY))
    return dict(n=len(M), mean_gross=M.gross.mean(), mean_turn=M.turnover.mean(), mean_net=M.net.mean(),
                ci_lo=ci[0], ci_hi=ci[1], q=ci[2], ci_lo_gross=cig[0], era_net=era.net.mean() if len(era) else np.nan,
                yrs_pos=int((yr > 0).sum()), breakeven=(M.gross.mean() / M.turnover.mean()) if M.turnover.mean() > 0 else np.nan,
                per_year=yr)


def calendar_consts(cal):
    global T_LO, T_HI, YEAR_T, YIDX
    T_LO = int(np.searchsorted(cal, IS_START)) - 1                 # signal bar; entry bar = T_LO + 1 >= IS_START
    T_HI = int(np.searchsorted(cal, PURGE, side='right')) - 2      # entry bar T_HI + 1 <= PURGE
    assert T_LO >= 0 and cal[T_LO + 1] >= IS_START and cal[T_HI + 1] <= PURGE
    YEAR_T = cal.year.to_numpy()
    YIDX = np.where((YEAR_T >= Y0) & (YEAR_T < Y0 + NY), YEAR_T - Y0, -1).astype(np.int64)


def read_assets():
    """ASSET_FILE with the reviewed ASSET_OVERRIDES applied."""
    A = pd.read_csv(ASSET_FILE).fillna({'name': ''})
    A['asset_class'] = [ASSET_OVERRIDES.get(t, c) for t, c in zip(A['ticker'], A['asset_class'])]
    return A


def asset_masks(tickers):
    A = read_assets().set_index('ticker').reindex(tickers)
    u1 = set(pd.read_csv(U1_FILE)['ticker'])
    is_u1 = np.array([t in u1 for t in tickers])
    is_stock = A['asset_class'].isin(['Stock', 'Unknown']).to_numpy() & ~is_u1
    return is_stock, is_u1


def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()



# ---------------------------------------------------------------- S4b event study
def etf_ctx(O, H, L, C, V):
    """Indicators, trend and triggers on the 12-ETF panel (T x 12); N x T copies for the simulator."""
    I = indicators(O, H, L, C, V)
    C1, C2, C3 = lag(C, 1), lag(C, 2), lag(C, 3)
    trend = (C > I['S200']) & (I['S50'] > I['S200'])
    TRIG = {'T1': I['RSI'] < 10, 'T2': (C < C1) & (C1 < C2) & (C2 < C3), 'T3': C < I['LOW20p']}
    NT = {k: np.ascontiguousarray(x.T) for k, x in dict(O=O, H=H, L=L, C=C, S5=I['S5'], RSI=I['RSI']).items()}
    return dict(trend=trend, TRIG=TRIG, NT=NT, O=O, C=C)


def baseline_table(ctx, trig, max_len=20):
    """Mean return over every other uptrend day of the same ETF (trend true, THIS trigger false; signal day in the IS
    range), entry at the next open, exit L bars later at the open (xt 0) or close (xt 1). Shape (N, max_len+1, 2)."""
    O, C = ctx['O'], ctx['C']; T, N = C.shape
    B = ctx['trend'] & ~ctx['TRIG'][trig]
    out = np.full((N, max_len + 1, 2), np.nan)
    for k in range(N):
        s = np.nonzero(B[T_LO:T_HI + 1, k])[0] + T_LO; e = s + 1
        for L_ in range(max_len + 1):
            x = e + L_; ok = x < T
            for xt, PX in ((0, O), (1, C)):
                r = PX[x[ok], k] / O[e[ok], k] - 1
                out[k, L_, xt] = np.nanmean(r) if np.isfinite(r).any() else np.nan
    return out


def run_trial(ctx, trig, outcome, seed=0):
    """Non-overlapping events per ETF (the simulator's one-position rule), excess = event gross - same-ETF baseline of
    the SAME length and exit price type - COST (costs on the event only; baseline gross)."""
    name, mode, hold = outcome
    sig = ctx['trend'] & ctx['TRIG'][trig]; nt = ctx['NT']
    k, t, x, r, xt = run_rule(np.ascontiguousarray(sig.T), nt['O'], nt['H'], nt['L'], nt['C'], nt['S5'], nt['RSI'],
                              mode, hold, False, 0.0, T_LO, T_HI)
    BT = baseline_table(ctx, trig)
    Lr = x - (t + 1)
    base = BT[k, Lr, xt]
    gross_exc = r - base; exc = gross_exc - COST
    ok = np.isfinite(exc); k, t, x, r, exc, gross_exc = k[ok], t[ok], x[ok], r[ok], exc[ok], gross_exc[ok]
    ent = CAL[t + 1]; yr = ent.year.to_numpy(); mo = np.asarray(ent.to_period('M').astype(str))
    out = dict(trig=trig, outcome=name, n=len(exc), k=k, t=t, exc=exc, year=yr, month=mo)
    if len(exc) < 20:
        out.update(mean=np.nan, q=np.nan, mean_gross=np.nan, q_gross=np.nan, verdict=False); return out
    lo_, hi_, q_ = year_ci(exc, mo, BOOT_REPS, seed, CI_LEVEL)          # month-block bootstrap (NET: the verdict)
    q_g = year_ci(gross_exc, mo, BOOT_REPS, seed, CI_LEVEL)[2]          # GROSS: what the martingale check centres
    yt = pd.DataFrame({'year': yr, 'exc': exc}).groupby('year')['exc'].agg(['size', 'mean']).reindex(range(Y0, Y0 + NY))
    qual = yt[yt['size'] >= MIN_EVENTS_YEAR]
    n_q, n_pos = len(qual), int((qual['mean'] > 0).sum())
    lo_e, hi_e = ERAS[RECENT]; era = (yr >= lo_e) & (yr <= hi_e)
    era_m = float(exc[era].mean()) if era.any() else np.nan
    verdict = bool(np.isfinite(era_m) and era_m > 0 and n_q >= MIN_QUAL_YEARS and n_pos >= math.ceil(2 * n_q / 3)
                   and q_ >= Q_PASS)
    out.update(mean=float(exc.mean()), mean_gross=float(gross_exc.mean()), q_gross=q_g, breakeven=float(gross_exc.mean()),
               avg_gross=float(r.mean()), ci_lo=lo_, ci_hi=hi_, q=q_, era=era_m, n_qual=n_q, n_pos_qual=n_pos,
               avg_bars=float((x - t).mean()), per_year=yt, verdict=verdict)
    return out


@njit(cache=True)
def gen_world(seed, mkt, sd_i, first, last, P0, M, on_frac, plant_rev, rev_thr, plant_vol, vspike):
    """Pack-2 generator (intrabar paths, martingale idio, market drift); copied verbatim."""
    np.random.seed(seed)
    T = mkt.shape[0]; N = sd_i.shape[0]
    O = np.full((N, T), np.nan); Hh = np.full((N, T), np.nan); Ll = np.full((N, T), np.nan); Cc = np.full((N, T), np.nan)
    for k in range(N):
        if first[k] < 0: continue
        so = sd_i[k] * np.sqrt(on_frac); si = sd_i[k] * np.sqrt((1.0 - on_frac) / M)
        prev = P0[k]; cr = 0; cv = 0
        for t in range(first[k], last[k] + 1):
            drift = 0.0
            if cr > 0:
                drift += plant_rev; cr -= 1
            if cv > 0:
                drift += plant_vol; cv -= 1
            o = prev * np.exp(mkt[t, 0] + so * np.random.standard_normal() - 0.5 * so * so)
            p = o; hi = o; lo = o
            for j in range(1, M + 1):
                p *= np.exp(mkt[t, j] + si * np.random.standard_normal() - 0.5 * si * si + drift / M)
                if p > hi: hi = p
                if p < lo: lo = p
            O[k, t] = o; Hh[k, t] = hi; Ll[k, t] = lo; Cc[k, t] = p
            if plant_rev > 0.0 and np.log(p / prev) < -rev_thr[k]: cr = 3
            if plant_vol > 0.0 and vspike[k, t]: cv = 20
            prev = p
    return O, Hh, Ll, Cc


def append_log(path, df, key='config_id'):
    """Append-only; an identical config already logged is not a new trial."""
    if path.exists():
        old = pd.read_csv(path, dtype={key: str}); new = df[~df[key].isin(old[key])]
        pd.concat([old, new], ignore_index=True).to_csv(path, index=False); return len(new), len(old) + len(new)
    df.to_csv(path, index=False); return len(df), len(df)



# ---------------------------------------------------------------- S4b drill
def drill_baseline(ctx, base_mask, max_len=10):
    """Mean return of every baseline window (signal day in IS, base_mask true; entry next open), by length L and exit
    price type (0 open, 1 close). Shape (N, max_len+1, 2). Vectorised over L."""
    O, C = ctx['O'], ctx['C']; T, N = C.shape; Ls = np.arange(max_len + 1)
    out = np.full((N, max_len + 1, 2), np.nan)
    for k in range(N):
        s = np.nonzero(base_mask[T_LO:T_HI + 1, k])[0] + T_LO
        if not len(s): continue
        e = s + 1; X = e[:, None] + Ls[None, :]; ok = X < T; Xc = np.where(ok, X, 0)
        o_e = O[e, k][:, None]
        for xt, PX in ((0, O), (1, C)):
            r = np.where(ok, PX[Xc, k] / o_e - 1, np.nan)
            with np.errstate(all='ignore'):
                out[k, :, xt] = np.nanmean(r, axis=0)
    return out


def drill_trial(ctx, exit_, elig=None, seed=0, names=None):
    """Frozen T3 rule (R1): uptrend + T3 [+ eligibility], next-open entry, X1/X2 exits with 10-bar time stop,
    non-overlapping; excess = gross - same-ticker baseline (same length + exit price type) - COST."""
    name, mode, hold = exit_
    el = np.ones_like(ctx['trend']) if elig is None else elig
    T3 = ctx['TRIG']['T3']; sig = ctx['trend'] & T3 & el; base = ctx['trend'] & ~T3 & el
    nt = ctx['NT']
    k, t, x, r, xt = run_rule(np.ascontiguousarray(sig.T), nt['O'], nt['H'], nt['L'], nt['C'], nt['S5'], nt['RSI'],
                              mode, hold, False, 0.0, T_LO, T_HI)
    BT = drill_baseline(ctx, base)
    g = r - BT[k, x - (t + 1), xt]; ok = np.isfinite(g)
    k, t, x, r, g = k[ok], t[ok], x[ok], r[ok], g[ok]; exc = g - COST
    ent = CAL[t + 1]; yr = ent.year.to_numpy(); mo = np.asarray(ent.to_period('M').astype(str))
    out = dict(exit=name, n=len(exc), k=k, t=t, x=x, gross=r, gexc=g, exc=exc, year=yr, month=mo)
    if len(exc) < 20:
        out.update(ok=False); return out
    lo, hi, q = year_ci(exc, mo, BOOT_REPS, seed, CI_LEVEL)
    glo, ghi, gq = year_ci(g, mo, BOOT_REPS, seed, CI_LEVEL)
    a, b = ERAS[RECENT]; era = (yr >= a) & (yr <= b)
    out.update(ok=True, mean=float(exc.mean()), lo=lo, hi=hi, q=q, era=float(exc[era].mean()) if era.any() else np.nan,
               gmean=float(g.mean()), glo=glo, ghi=ghi, gq=gq, gera=float(g[era].mean()) if era.any() else np.nan,
               avg_gross=float(r.mean()), avg_bars=float((x - t).mean()), breakeven=float(g.mean()))
    return out


def r2_pass(res, gross=False):
    """R2: BOTH exits q >= 95 AND CI90 lower bound > 0, AND 2019-2023 mean > 0 (net; gross for the martingale check)."""
    q_, lo_, era_ = ('gq', 'glo', 'gera') if gross else ('q', 'lo', 'era')
    return bool(all(o.get('ok') and o[q_] >= Q_DRILL and o[lo_] > 0 and np.isfinite(o[era_]) and o[era_] > 0
                    for o in res.values()))


def make_world_for(Creal, seed, plant=False):
    """Pack-2 generator on any panel: real listing spans + gaps, market drift, intrabar paths, martingale idio."""
    T_, N_ = Creal.shape; valid = np.isfinite(Creal)
    first = np.where(valid.any(0), valid.argmax(0), -1).astype(np.int64)
    last = np.where(valid.any(0), T_ - 1 - valid[::-1].argmax(0), -1).astype(np.int64)
    p0 = np.array([Creal[first[i], i] if first[i] >= 0 else 1.0 for i in range(N_)])
    with np.errstate(all='ignore'):
        sd_ = np.nanstd(np.diff(np.log(Creal), axis=0), axis=0)
    sd_ = np.where(np.isfinite(sd_), np.clip(sd_, 0.005, 0.10), 0.02)
    sdi = np.sqrt(np.maximum(sd_ ** 2 - MKT_SD ** 2, (0.25 * sd_) ** 2))
    rng = np.random.default_rng(30_000 + seed)
    mkt = np.empty((T_, M_STEPS + 1))
    mkt[:, 0] = rng.normal(MKT_MU * ON_FRAC - 0.5 * MKT_SD ** 2 * ON_FRAC, MKT_SD * np.sqrt(ON_FRAC), T_)
    s_in = MKT_SD * np.sqrt((1 - ON_FRAC) / M_STEPS)
    mkt[:, 1:] = rng.normal(MKT_MU * (1 - ON_FRAC) / M_STEPS - 0.5 * s_in ** 2, s_in, (T_, M_STEPS))
    O, Hh, Ll, Cc = gen_world(seed, mkt, sdi, first, last, p0, M_STEPS, ON_FRAC, DRILL_PLANT if plant else 0.0,
                              2 * sd_, 0.0, np.zeros((N_, T_), bool))
    W = {c: np.ascontiguousarray(z.T) for c, z in zip('OHLC', (O, Hh, Ll, Cc))}
    for c in 'OHLC': W[c][~valid] = np.nan
    return W


def tiingo_get(url):
    for i in range(4):
        try:
            return json.load(urllib.request.urlopen(url, timeout=60))
        except Exception as e:
            if i == 3: raise
            time.sleep(2 + 3 * i)

In [ ]:
# ===== Cell 2 — corpus panels (cut at load; pack-2 cache), SPY regime =====
t0 = time.time()
TICKERS, CAL, P = load_panels(); calendar_consts(CAL)
assert CAL.max() <= CUT_DATE
i_spy = TICKERS.index('SPY'); spyC = P['C'][:, i_spy]
spy_s200 = roll(spyC[:, None], 200)[:, 0]
SPY_BELOW = spyC < spy_s200                                   # signal-day regime (R3)
yb = pd.Series(SPY_BELOW[np.isfinite(spy_s200)], index=CAL[np.isfinite(spy_s200)]).groupby(lambda d: d.year).mean()
WEAK_YEARS = sorted(int(y) for y, v in yb.items() if v >= 0.25 and Y0 <= y < Y0 + NY)
print(f'corpus {len(TICKERS)} x {len(CAL)} [{time.time()-t0:.0f}s] | SPY-weak years (>= 25% of days below SMA200): {WEAK_YEARS}')

## Step 0 — PRIMARY universe by rule (R5a)

In [ ]:
# ===== Cell 3 — build the 60-ETF universe (cached pulls; rule only, no outcomes) =====
token = tiingo_token(); assert token, 'TIINGO_API_KEY needed'
rule_now = json.load(open(UNIV_RULE_FILE)).get('rule_version') if UNIV_RULE_FILE.exists() else ('v1' if UNIV_FILE.exists() else None)
if UNIV_FILE.exists() and rule_now != UNIV_RULE_VERSION:      # superseded list (never approved/frozen): keep for the record
    UNIV_FILE.rename(UNIV_FILE.with_name(f'replication_universe_{rule_now}_superseded.csv'))
    print(f'rule changed {rule_now} -> {UNIV_RULE_VERSION}: rebuilding (superseded list kept)')
if not UNIV_FILE.exists():
    st_f = DCACHE/'supported_tickers.csv'
    if not st_f.exists():
        raw = urllib.request.urlopen('https://apimedia.tiingo.com/docs/tiingo/daily/supported_tickers.zip', timeout=180).read()
        zf = zipfile.ZipFile(io.BytesIO(raw)); pd.read_csv(zf.open(zf.namelist()[0])).to_csv(st_f, index=False)
    st = pd.read_csv(st_f)
    st['ticker'] = st['ticker'].astype(str).str.upper()
    n_rows = st.groupby('ticker').size()                      # > 1 row = reused ticker / mixed histories
    sd_, ed_ = pd.to_datetime(st['startDate'], errors='coerce'), pd.to_datetime(st['endDate'], errors='coerce')
    pool = st[(st.assetType == 'ETF') & st.exchange.isin(US_EXCH) & (st.priceCurrency == 'USD')
              & (sd_ <= '2013-12-31') & (ed_ >= '2013-12-31') & ~st.ticker.isin(U1_ETFS)
              & st.ticker.str.fullmatch(r'[A-Z]+')].drop_duplicates('ticker')
    print(f'pool: {len(pool)} ETFs (US-listed, USD, trading by 2014-01-01, excl. U1)')
    # 2013 bars per candidate (cached): raw close x raw volume for $vol; adjClose for returns
    c13 = DCACHE/'bars_2013.parquet'
    have = pd.read_parquet(c13) if c13.exists() else pd.DataFrame(columns=['ticker', 'date', 'close', 'volume', 'adjClose'])
    todo = sorted(set(pool.ticker) - set(have.ticker)); rows = []; t0 = time.time()
    for i, tk in enumerate(todo):
        try:
            js = tiingo_get(f'https://api.tiingo.com/tiingo/daily/{tk}/prices?startDate=2012-12-01&endDate=2013-12-31&token={token}')
            rows += [dict(ticker=tk, date=j['date'][:10], close=j['close'], volume=j['volume'], adjClose=j['adjClose']) for j in js]
        except Exception as e:
            rows.append(dict(ticker=tk, date=None, close=np.nan, volume=np.nan, adjClose=np.nan))
        if (i + 1) % 200 == 0:
            have = pd.concat([have, pd.DataFrame(rows)]); have.to_parquet(c13); rows = []
            print(f'  2013 bars: {i+1}/{len(todo)} [{time.time()-t0:.0f}s]')
    have = pd.concat([have, pd.DataFrame(rows)]); have.to_parquet(c13)
    B13 = have.dropna(subset=['date']); B13['date'] = pd.to_datetime(B13['date'])
    B13 = B13[B13.date.dt.year == RANK_YEAR]
    agg = B13.assign(dv=B13.close * B13.volume).groupby('ticker').agg(n=('dv', 'size'), dvol=('dv', 'median'))
    ranked = agg[agg.n >= MIN_BARS_2013].sort_values('dvol', ascending=False)
    # U1 2013 returns (corpus, adjusted) for the near-duplicate screen
    u1r = pd.DataFrame({e: pd.Series(P['C'][:, TICKERS.index(e)], index=CAL) for e in U1_ETFS if e in TICKERS}).pct_change()
    u1r = u1r[u1r.index.year == RANK_YEAR]
    meta_f = DCACHE/'meta.json'; META = json.load(open(meta_f)) if meta_f.exists() else {}
    keep, excl, n_country = [], [], 0
    for tk, row in ranked.iterrows():
        if len(keep) >= N_ETF: break
        if tk not in META:
            try:
                m_ = tiingo_get(f'https://api.tiingo.com/tiingo/daily/{tk}?token={token}')
                META[tk] = dict(name=m_.get('name') or '', startDate=m_.get('startDate'), endDate=m_.get('endDate'))
            except Exception:
                META[tk] = dict(name='', startDate=None, endDate=None)
            json.dump(META, open(meta_f, 'w'))
        nm = META[tk]['name']; sd_m = pd.to_datetime(META[tk]['startDate'], errors='coerce')
        if n_rows.get(tk, 0) > 1: excl.append((tk, nm, 'ticker reuse: multiple Tiingo rows')); continue
        if not FUND_RE.search(nm): excl.append((tk, nm, 'not a fund (current name has no fund marker)')); continue
        if not (pd.notna(sd_m) and sd_m <= pd.Timestamp('2013-01-02')):
            excl.append((tk, nm, f'current fund starts {META[tk]["startDate"]} (2013 history not this fund)')); continue
        if ETN_RE.search(nm): excl.append((tk, nm, 'ETN (debt note)')); continue
        if LEV_RE.search(nm): excl.append((tk, nm, 'leveraged/inverse/vol')); continue
        if NONEQ_RE.search(nm): excl.append((tk, nm, 'non-equity')); continue
        r = B13[B13.ticker == tk].set_index('date')['adjClose'].sort_index().pct_change()
        cm = u1r.apply(lambda col: r.corr(col)).max()
        if np.isfinite(cm) and cm > CORR_MAX: excl.append((tk, nm, f'U1 near-duplicate (corr {cm:.3f})')); continue
        is_c = bool(COUNTRY_RE.search(nm))
        if is_c and n_country >= MAX_COUNTRY: excl.append((tk, nm, 'country/region cap')); continue
        n_country += is_c
        keep.append(dict(rank=len(keep) + 1, ticker=tk, name=nm, underlying='non-US' if is_c else 'US',
                         dvol_2013=row.dvol, max_corr_u1=round(float(cm), 3) if np.isfinite(cm) else np.nan))
    U = pd.DataFrame(keep); U.to_csv(UNIV_FILE, index=False)
    json.dump(dict(rule_version=UNIV_RULE_VERSION, built_utc=pd.Timestamp.now(tz='UTC').isoformat()), open(UNIV_RULE_FILE, 'w'))
    pd.DataFrame(excl, columns=['ticker', 'name', 'reason']).to_csv(DRILL_DIR/'replication_excluded.csv', index=False)
U = pd.read_csv(UNIV_FILE); EX = pd.read_csv(DRILL_DIR/'replication_excluded.csv')
pd.set_option('display.max_rows', 300); pd.set_option('display.width', 220)
print(U.assign(dvol_2013=(U.dvol_2013 / 1e6).round(1)).to_string(index=False))
print(f'\n{len(U)} ETFs | non-US underlying {int((U.underlying == "non-US").sum())} (cap {MAX_COUNTRY}) | '
      f'excluded while filling: {EX.reason.str.split(":| \\(").str[0].value_counts().to_dict()}')
print(EX.to_string(index=False))
UNIV_SHA_NOW = sha(UNIV_FILE); print(f'\nreplication_universe.csv sha256 = {UNIV_SHA_NOW}')
assert UNIVERSE_APPROVED, 'Review the list, set UNIVERSE_APPROVED = True, and paste the sha256 back for the freeze commit.'
assert UNIVERSE_SHA and UNIVERSE_SHA == UNIV_SHA_NOW, 'UNIVERSE_SHA must be the frozen value from the R5 freeze commit.'

In [ ]:
# ===== Cell 4 — full adjusted histories for the 60 (endDate = cut) -> ETF panel; stock-universe inputs =====
for tk in U.ticker:
    f_ = REP_DIR/f'{tk}.csv'
    if f_.exists(): continue
    js = pd.DataFrame(tiingo_get(f'https://api.tiingo.com/tiingo/daily/{tk}/prices?startDate=2009-06-01'
                                 f'&endDate={CUT_DATE:%Y-%m-%d}&token={token}'))
    pd.DataFrame({'Date': pd.to_datetime(js['date']).dt.strftime('%Y-%m-%d'), 'Open': js['adjOpen'], 'High': js['adjHigh'],
                  'Low': js['adjLow'], 'Close': js['adjClose'], 'Volume': js['adjVolume']}).to_csv(f_, index=False)
ETF = {c: np.full((len(CAL), len(U)), np.nan) for c in 'OHLCV'}
for j, tk in enumerate(U.ticker):
    d = load_px(REP_DIR/f'{tk}.csv'); pos = CAL.get_indexer(d['Date']); ok = pos >= 0
    for c, col in zip('OHLCV', ['Open', 'High', 'Low', 'Close', 'Volume']):
        ETF[c][pos[ok], j] = d[col].to_numpy(float)[ok]
bad = ~(ETF['C'] > 0)
for c in 'OHLC': ETF[c][bad] = np.nan
assert np.isfinite(ETF['C']).any(0).all(), 'an ETF has no bars <= cut'
IS_STOCK, IS_U1 = asset_masks(TICKERS)
print(f'ETF panel {ETF["C"].shape} | stock-universe pool {int(IS_STOCK.sum())} (U2 = top {TOP_U2} by trailing 60-bar $vol)')


def stock_ctx(O, H, L, C, V):
    ctx = etf_ctx(O, H, L, C, V)
    ctx['U2'] = universes(lag(roll(C * V, 60), 1), IS_STOCK, IS_U1)['U2']
    return ctx

## Part A — validate the null on the martingale world, BOTH universes (R5d)

In [ ]:
# ===== Cell 5 — 16 seeds x 2 universes + power =====
def run_universe(univ, seed, plant=False):
    if univ == 'ETF60':
        W = make_world_for(ETF['C'], seed, plant); ctx = etf_ctx(W['O'], W['H'], W['L'], W['C'], ETF['V']); el = None
    else:
        W = make_world_for(P['C'], seed, plant); ctx = stock_ctx(W['O'], W['H'], W['L'], W['C'], P['V']); el = ctx['U2']
    return {ex[0]: drill_trial(ctx, ex, el, seed) for ex in DRILL_EXITS}


VAL = {}
for univ in ['ETF60', 'STK300']:
    rec = []; t0 = time.time()
    for s in range(DRILL_SEEDS):
        res = run_universe(univ, s)
        for ex, o in res.items():
            rec.append(dict(seed=s, exit=ex, n=o['n'], gmean=o.get('gmean'), gq=o.get('gq')))
        rec[-1]['r2_gross'] = rec[-2]['r2_gross'] = r2_pass(res, gross=True)
    R_ = pd.DataFrame(rec); R_.to_csv(DRILL_DIR/f'protocol_{univ}.csv', index=False)
    rows = {}
    for ex, g in R_.groupby('exit'):
        ps = g.set_index('seed')
        rows[ex] = dict(n_med=int(g.n.median()), mean_gexc=ps.gmean.mean(),
                        t_exc=float(stats.ttest_1samp(ps.gmean, 0.0).statistic),
                        mean_q=ps.gq.mean(), t_q=float(stats.ttest_1samp(ps.gq, 50.0).statistic))
    F = pd.DataFrame(rows).T
    false_pass = float(R_.groupby('seed').r2_gross.first().mean())
    centring = bool((F.t_exc.abs() < 2.5).all() and F.mean_q.between(40, 60).all() and (F.t_q.abs() < 2.5).all())
    null_ok = bool(centring and false_pass <= FALSE_PASS_MAX)
    print(f'\n[{univ}] {DRILL_SEEDS} seeds [{time.time()-t0:.0f}s]'); print(F.round(4).to_string())
    print(f'  gross R2 false-pass rate {false_pass:.1%} (<= {FALSE_PASS_MAX:.0%}) | centring {"PASS" if centring else "FAIL"}')
    det = []
    if null_ok:
        for s in range(DRILL_POWER_SEEDS):
            res = run_universe(univ, 1000 + s, plant=True)
            det.append(bool(all(o.get('ok') and o['gq'] >= Q_DRILL for o in res.values())))
            print(f'  power seed {1000+s}: ' + ' '.join(f'{ex} q={o.get("gq", np.nan):.1f}' for ex, o in res.items()))
    power_ok = bool(null_ok and sum(det) >= math.ceil(0.75 * DRILL_POWER_SEEDS))
    VAL[univ] = dict(families=json.loads(F.to_json(orient='index')), false_pass=false_pass, centring=centring,
                     null_ok=null_ok, power_det=det, power_ok=power_ok, ok=bool(null_ok and power_ok))
    print(f'  [{univ}] null {"PASS" if null_ok else "FAIL"} | power {"PASS" if power_ok else ("FAIL" if null_ok else "not run")}'
          f' -> real data {"ALLOWED" if VAL[univ]["ok"] else "STOPPED"}')
json.dump(dict(written_utc=pd.Timestamp.now(tz='UTC').isoformat(), n_seeds=DRILL_SEEDS, universe_sha=UNIV_SHA_NOW, **VAL),
          open(DRILL_PROTO, 'w'), indent=1, default=float)
print('\nSUMMARY drill validation:', {u: ('PASS' if v['ok'] else 'FAIL') for u, v in VAL.items()})

## Part B — frozen rule on real data (gated per universe)

In [ ]:
# ===== Cell 6 — real data =====
PRO = json.load(open(DRILL_PROTO))
assert PRO['universe_sha'] == UNIVERSE_SHA == sha(UNIV_FILE), 'universe changed since validation'
assert PRO['n_seeds'] >= 16 or os.environ.get('PACK2_TEST') == '1'
pct = lambda v: f'{v:+.2%}' if v is not None and np.isfinite(v) else 'nan'
REAL = {}
for univ in ['ETF60', 'STK300']:
    if not PRO[univ]['ok']:
        print(f'[{univ}] STOPPED by its validation — not evaluated'); continue
    if univ == 'ETF60':
        ctx = etf_ctx(ETF['O'], ETF['H'], ETF['L'], ETF['C'], ETF['V']); el, labels = None, np.array(U.ticker)
    else:
        ctx = stock_ctx(P['O'], P['H'], P['L'], P['C'], P['V']); el, labels = ctx['U2'], np.array(TICKERS)
    res = {ex[0]: drill_trial(ctx, ex, el, 41) for ex in DRILL_EXITS}
    REAL[univ] = (res, labels)
    tag = '  [SURVIVORSHIP-BIASED IN FAVOUR OF DIP-BUYING]' if univ == 'STK300' else '  [PRIMARY]'
    print(f'\n===== {univ}{tag}')
    for ex, o in res.items():
        print(f'  T3 {ex}: n={o["n"]} | avg bars {o.get("avg_bars", np.nan):.1f} | avg gross {pct(o.get("avg_gross"))} | '
              f'excess net {pct(o.get("mean"))} CI90 [{pct(o.get("lo"))}, {pct(o.get("hi"))}] q {o.get("q", np.nan):.1f} | '
              f'era 2019-23 {pct(o.get("era"))} | break-even {pct(o.get("breakeven"))}')
    print(f'  R2 ({univ}): {"PASS" if r2_pass(res) else "FAIL"}')
    py = pd.concat({ex: pd.Series(o['exc']).groupby(o['year']).agg(['size', 'mean']) for ex, o in res.items()}, axis=1)
    print('  per year (n, mean excess):'); print(py.round(4).to_string())
    pe = pd.concat({ex: pd.Series(o['exc']).groupby(labels[o['k']]).agg(['size', 'mean']) for ex, o in res.items()}, axis=1)
    if univ == 'ETF60':
        print('  per ETF (n, mean excess):'); print(pe.round(4).to_string())
    else:
        print('  per stock — distribution across stocks (n, mean excess):'); print(pe.describe().round(4).to_string())
    py.to_csv(DRILL_DIR/f'real_{univ}_per_year.csv'); pe.to_csv(DRILL_DIR/f'real_{univ}_per_ticker.csv')
    for ex, o in res.items():
        below = SPY_BELOW[o['t']]; weak = np.isin(o['year'], WEAK_YEARS)
        line = (f'  regime {ex}: SPY>200d n={int((~below).sum())} {pct(o["exc"][~below].mean() if (~below).any() else np.nan)} | '
                f'SPY<200d n={int(below.sum())} {pct(o["exc"][below].mean() if below.any() else np.nan)} | '
                f'SPY-weak years n={int(weak.sum())} {pct(o["exc"][weak].mean() if weak.any() else np.nan)}')
        if univ == 'ETF60':
            us = (U.underlying.to_numpy()[o['k']] == 'US')
            line += (f' | US n={int(us.sum())} {pct(o["exc"][us].mean() if us.any() else np.nan)} | '
                     f'non-US n={int((~us).sum())} {pct(o["exc"][~us].mean() if (~us).any() else np.nan)}')
        print(line + '   (descriptive, outside the pass)')

In [ ]:
# ===== Cell 7 — pre-registered reading (R5c) + trial log =====
prim = r2_pass(REAL['ETF60'][0]) if 'ETF60' in REAL else None
sec = r2_pass(REAL['STK300'][0]) if 'STK300' in REAL else None
if prim is None: reading = 'PRIMARY not evaluated (validation failed) -> no replication evidence'
elif prim: reading = 'PRIMARY PASS -> primary evidence: the T3 rule replicates (lockbox may be opened ONCE, R4)'
elif sec: reading = 'PRIMARY FAIL, SECONDARY PASS -> likely survivorship: does not replicate'
else: reading = 'PRIMARY FAIL' + (', SECONDARY FAIL' if sec is not None else '') + ' -> does not replicate'
print('READING (R5c):', reading)
now = pd.Timestamp.now(tz='UTC').isoformat(); rows, rets = [], []
for univ, (res, _) in REAL.items():
    for ex, o in res.items():
        cid = f's4bdrill|T3|{univ}|{ex}'; m = pd.Series(o['exc']).groupby(o['month']).mean() if o['n'] else pd.Series(dtype=float)
        rows.append(dict(kind='trial', config_id=cid, pack='s4bdrill', screen='S4b-T3', universe=univ, rule=f'T3_{ex}',
                         n=o['n'], n_months=len(m), WR=float((o['exc'] > 0).mean()) if o['n'] else np.nan, avg_net=o.get('mean'),
                         sd=m.std(), skew=m.skew(), kurt=m.kurt(), sr_monthly=m.mean() / m.std() if m.std() > 0 else np.nan,
                         pct_null=o.get('q'), verdict=r2_pass(res), backfilled=False,
                         note=('survivorship-biased (stocks)' if univ == 'STK300' else 'primary replication') + '; frozen T3 rule',
                         logged_at=now))
        rets += [dict(config_id=cid, month=mm, ret=v) for mm, v in m.items()]
if rows:
    LOG = pd.DataFrame(rows); RET = pd.DataFrame(rets); RET['key'] = RET.config_id + '|' + RET.month
    n_new, _ = append_log(TRIAL_LOG, LOG); append_log(TRIAL_RET, RET, key='key')
    print(f'trial log: +{n_new} new | cumulative trials {int((pd.read_csv(TRIAL_LOG).kind == "trial").sum())}')